# MoveFormer baseline demo

This notebook is a minimal demo of the MoveBench MoveFormer baseline APIs.

It reuses:
- MoveBench iSSF **parametric point stores** (candidate sets + env samples), and
- MoveFormer upstream `SimpleTrajectoryProcessor.encode()` (time/location/movement encoding).

Prereqs (once):
- Install optional deps: `pip install -e '.[moveformer]'`
- Install a compatible `torch` build for your system.

In [1]:
import torch
from torch.utils.data import DataLoader

from movebench import Study
from movebench.models.moveformer import build_moveformer_selection_model
from movebench.train import collate_moveformer, eval_moveformer, fit_moveformer

In [2]:
study_id = 'tsavo_lions'  # e.g. Study.list_available()[0]
layers_config = '../configs/layers_local.yaml'
device = torch.device('cpu')

# iSSF-style sampler settings (P=101 total candidates)
sampler = dict(
    num_candidates=100,
    random_seed=0,
    step_multiplier=2.0,
    step_q=0.95,
    gap_q=0.999,
    bounds_buffer_deg=0.01,
)


In [3]:
study = Study(study_id)

# Ensure rasters exist (download if missing), then extract a parametric point store.
# MoveFormer baseline reuses this iSSF point store.
# Uncomment to (re)create feature stores.

# try:
#     rep = study.extract_point_features(
#         config=layers_config,
#         splits=['train', 'test'],
#         parametric=True,
#         n_workers=16,
#         skip_existing=True,
#         **sampler,
#     )
# except FileNotFoundError as e:
#     if 'rasters' not in str(e):
#         raise
#     study.download_rasters(
#         config=layers_config,
#         splits=['train', 'test'],
#         n_workers=16,
#         skip_existing=True,
#     )
#     rep = study.extract_point_features(
#         config=layers_config,
#         splits=['train', 'test'],
#         parametric=True,
#         n_workers=16,
#         skip_existing=True,
#         **sampler,
#     )

In [4]:
train_ds = study.moveformer_dataset(
    split='train',
    config=layers_config,
    max_len=500,
    min_len=2,
    **sampler,
)
test_ds = study.moveformer_dataset(
    split='test',
    config=layers_config,
    max_len=500,
    min_len=2,
    **sampler,
)

# Note: MoveFormerPointStoreDataset keeps the whole point-store tensors in memory.
# Using num_workers>0 may duplicate memory on platforms that use spawn.
train_loader = DataLoader(
    train_ds,
    batch_size=24,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_moveformer,
    drop_last=False,
)
test_loader = DataLoader(
    test_ds,
    batch_size=24,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_moveformer,
    drop_last=False,
)

len(train_ds), len(test_ds), train_ds.num_candidates_total, train_ds.cov_dim

/Users/justinkay/miniconda3/envs/movebench/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/justinkay/movebench/movebench/datasets/moveformer.py:170: RuntimeWarning: Multiple matching point feature stores found even though the sampler was fully specified; choosing the most recently created store: /Users/justinkay/movebench/data/movebench/studies_10ind/tsavo_lions/features/points/train/dc5e291c4989ce94/7ccdb408c6674ad1. To avoid this warning, delete older matching stores under the same config_hash directory.
  store_dir, m = select_point_store_dir(
/Users/justinkay/movebench/movebench/trajectories.py:105: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future 

(10, 5, 101, 53)

In [5]:
batch = next(iter(train_loader))
sorted(batch.keys()), {k: tuple(v.shape) for k, v in batch.items()}

(['cov',
  'cov_cand',
  'dtime_cont',
  'location',
  'location_cand',
  'location_cand_mov',
  'location_mov',
  'location_tgt',
  'location_tgt_mov',
  'loss_mask',
  'mask',
  'taxon',
  'time_cont',
  'time_cont_tgt',
  'time_day',
  'time_day_tgt',
  'time_lmt',
  'time_lmt_tgt',
  'time_month',
  'time_month_tgt',
  'time_weekday',
  'time_weekday_tgt'],
 {'mask': (10, 500),
  'dtime_cont': (10, 500, 12),
  'time_cont': (10, 500, 10),
  'time_month': (10, 500),
  'time_day': (10, 500),
  'time_weekday': (10, 500),
  'time_lmt': (10, 500, 2),
  'location': (10, 500, 3),
  'time_cont_tgt': (10, 500, 10),
  'time_month_tgt': (10, 500),
  'time_day_tgt': (10, 500),
  'time_weekday_tgt': (10, 500),
  'time_lmt_tgt': (10, 500, 2),
  'location_tgt': (10, 500, 3),
  'location_cand': (10, 500, 101, 3),
  'location_mov': (10, 500, 2),
  'location_tgt_mov': (10, 500, 2),
  'location_cand_mov': (10, 500, 101, 2),
  'taxon': (10, 500, 100),
  'cov': (10, 500, 53),
  'cov_cand': (10, 500, 101

In [6]:
model = build_moveformer_selection_model(cov_dim=train_ds.cov_dim).to(device)

features = {k: v.to(device) for k, v in batch.items()}
logits, loss, metrics = model(features)

logits.shape, float(loss.item()), {k: float(v.item()) for k, v in metrics.items()}

(torch.Size([10, 500, 101]),
 1.0001262426376343,
 {'acc': 0.004758128430694342,
  'acc_top50': 0.4924663007259369,
  'acc_top5': 0.05471847578883171})

In [7]:
fit = fit_moveformer(
    model,
    train_loader,
    test_loader,
    device,
    max_steps=200,
    eval_interval=50,
    patience=10,
    seed=0,
)

train_metrics = eval_moveformer(model, train_loader, device)
test_metrics = eval_moveformer(model, test_loader, device)

fit, train_metrics, test_metrics

({'best_step': 49,
  'best_val_nll': 4.618007136511125,
  'train': {'nll': 4.613898474672879,
   'nce': 0.9997352090451546,
   'topk': {1: 0.007137192704203013,
    5: 0.05551149881046788,
    10: 0.12291831879460745,
    25: 0.26724821570182394,
    50: 0.5107057890563045},
   'num_items': 10,
   'num_tokens': 1261},
  'val': {'nll': 4.618007136511125,
   'nce': 1.0006254700520454,
   'topk': {1: 0.024721878862793572,
    5: 0.09394313967861558,
    10: 0.16069221260815822,
    25: 0.2830655129789864,
    50: 0.48825710754017304},
   'num_items': 5,
   'num_tokens': 809},
  'history': [{'step': 49.0,
    'train_nll': 4.613898087455393,
    'val_nll': 4.618007136511125},
   {'step': 99.0,
    'train_nll': 4.609317691812054,
    'val_nll': 4.619810880040945},
   {'step': 149.0,
    'train_nll': 4.598533684823552,
    'val_nll': 4.622380241328028},
   {'step': 199.0,
    'train_nll': 4.562833007038065,
    'val_nll': 4.622345234761279}]},
 {'nll': 4.613898474672879,
  'nce': 0.9997352090